# GISLR · Stage 1 — `StreamingGRU` benchmark

**Pipeline stage** for GISLR: per-subset feature caches → `StreamingGRU` training runs → registry run folders → canonical-eval handoff. A thin driver over the unified training stack in [`modules/model/`](modules/model/__init__.py) — model classes, data layer, training driver, registry and reporting are all shared with the sibling `gislr.1.model.*.ipynb` notebooks; **this notebook only sets the architecture identity (`ARCH`), the subsets and the hyperparameters**.

Trains the **top three subsets** from the discriminability comparison ([docs/daily/2026-07-16.md](../docs/daily/2026-07-16.md)) as controlled ablations (TODO §3.1) under training regime **v2-plateau-300** (TODO §3.2). GISLR is a competition dataset — grading/submission is a separate, later step (§7), so nothing here touches the Kaggle test server.

**Artifacts produced**

| artifact | path |
|---|---|
| per-subset feature caches (skip-if-exists, shared across architecture notebooks) | `data/cache/gislr/features/{train,val}_<tag>_{data,offsets}.npy` |
| run-dir pointer per subset (drives auto-resume) | `data/cache/runs/gislr_gru_<tag>.txt` |
| registry run folder, one per training start | `data/models/<run_id>/` (run_id = epoch seconds) |
| — the run record (schema: README.md § "meta.json schema") | `data/models/<run_id>/meta.json` |
| — checkpoints (gitignored) | `data/models/<run_id>/{best,last}.pt` |
| — plots, landmark indices, eval artifacts | `data/models/<run_id>/assets/` |
| queryable run index (rebuild after evals) | `data/models/index.csv` via `modules/scripts/build_model_index.py` |

**Resumability** — cache builds skip existing files and write atomically; training auto-resumes an *interrupted* run from `last.pt` (checkpoint + `meta.json` saved every epoch, early-stopping counter and LR-scheduler state included) via its pointer file. A **finished** run (early-stopped or epoch cap) is never reused: re-running §4 starts a fresh epoch-seconds run folder per subset. An interrupt never forces a full rerun.

**Design decisions**

- All reusable logic lives in `modules/model/` (TODO §0.3): `architectures.py` is the single model-class definition shared with the eval script — state_dicts can never drift.
- Training reports through **one progress bar per run** (batch progress, losses, LR, plateau counter all in the same bar) — no nested bars, no per-epoch print spam.
- Subset index lists come from the canonical registry [`modules/dataset/landmark/subsets.py`](modules/dataset/landmark/subsets.py) — never re-derived.
- Training regime **v2-plateau-300** (TODO §3.2): batch 512, lr 1e-3, ReduceLROnPlateau, epoch cap 300 with early stopping on val-acc plateau. Not hyperparameter-comparable to the retired v1-onecycle-60 family; the canonical eval split/metric is unchanged.
- Canonical per-class evaluation stays in `modules/scripts/eval_gru.py` (identical split/metric for every leaderboard run); §6 prints the exact command per run — **run by the user, after training**.

**Kernel requirements**: project `.venv`, CWD = `src/`, CUDA GPU.


## 1. Setup

All imports + every tunable. **`TRAIN_SUBSETS` is the primary knob** — the later cells each re-declare their own working list from it, so tweaking one section never requires re-running another. `COORDS = "xy"` is the z-drop ablation (TODO §3.1) and flows through cache tags, run folders and meta.json automatically.

In [ ]:
# ============================================================
# Setup — imports, architecture identity, tunables, resolved paths
# ============================================================
import matplotlib.pyplot as plt
import pandas as pd

from modules.dataset.landmark.subsets import get_subset
from modules.model import (
    eval_command, load_meta, pointer_run_dir, save_learning_curves,
    subset_tag, train_run, comparison_row,
)
from modules.paths import CACHE_DIR, MODELS_DIR, gislr_dir

COORDS = "xyz"   # coordinate channels fed to the model ("xy" drops z — TODO §3.1)

# architecture identity — the ONLY block that differs between the sibling
# gislr.1.model.*.ipynb notebooks (everything else lives in modules/model)
ARCH = "gru"   # key into modules.model.ARCHS -> StreamingGRU
#: top-3 subsets by probe_acc_global (docs/daily/2026-07-16.md) — THE knob of this notebook
TRAIN_SUBSETS = ["ME_126", "ME_132", "FP_118"]

# training regime v2 (TODO §3.2): bigger batch, epoch cap 300 with early
# stopping on val-accuracy plateau, ReduceLROnPlateau on the same signal.
TRAIN_REGIME = "v2-plateau-300"
HYP = dict(batch_size=512, lr=1e-3, hidden_size=256, num_layers=2,
           dropout=0.3, weight_decay=1e-4, epochs=300, grad_clip=5.0,
           lr_factor=0.5, lr_patience=5,       # ReduceLROnPlateau on val acc
           es_patience=15, es_min_delta=1e-3)  # early stop: no +0.1pt gain in 15 epochs
SOURCE = f"gislr.1.model.{ARCH}.ipynb"

# download only the GISLR competition data (never resolve_datasets() here —
# that would pull every dataset incl. POPSIGN)
DATA_DIR = gislr_dir()

def ptr_key(name: str) -> str:
    return f"gislr_{ARCH}_{subset_tag(name, COORDS)}"

print(f"DATA_DIR   = {DATA_DIR}")
print(f"CACHE_DIR  = {CACHE_DIR}")
print(f"MODELS_DIR = {MODELS_DIR} · arch {ARCH} · regime {TRAIN_REGIME}")
print("subsets to train: " + ", ".join(
    f"{n} ({len(get_subset(n))} lm)" for n in TRAIN_SUBSETS))

## 2. Canonical split check

Sanity-check the stratified 90/10 seed-42 split that every leaderboard run and `modules/scripts/eval_gru.py` share (85,029 train / 9,448 val). Purely diagnostic — the training driver calls `get_canonical_split()` itself.

In [ ]:
# ============================================================
# Canonical split sanity check — must match the leaderboard split exactly
# (stratified 90/10, random_state=42; the split itself is asserted in
# modules.model.data.get_canonical_split)
# ============================================================
from modules.model import get_canonical_split, load_label_map

sign2idx = load_label_map(DATA_DIR)
train_split, val_split = get_canonical_split(DATA_DIR, sign2idx)
print(f"train {len(train_split)} / val {len(val_split)} · "
      f"{train_split['sign'].nunique()} classes · "
      f"val classes {val_split['sign'].nunique()}")
display(train_split.head(3))

## 3. Feature caches (per subset)

One flat float32 array + offsets index per (split, subset, coords) under `data/cache/gislr/features/`, decoded from raw parquet once and reused by every later run and every sibling architecture notebook (skip-if-exists, atomic writes). Delete a file pair to force a rebuild.

In [ ]:
# ============================================================
# Build per-subset feature caches (tunable: CACHE_SUBSETS)
# Safe to re-run any time — existing caches are skipped. The training
# driver also builds missing caches itself; this cell just front-loads
# the ~minutes-long decode so training cells start instantly.
# ============================================================
from modules.model import build_subset_cache, get_canonical_split, load_label_map
from modules.model.data import FEATURES_DIR

CACHE_SUBSETS = TRAIN_SUBSETS

sign2idx = load_label_map(DATA_DIR)
train_split, val_split = get_canonical_split(DATA_DIR, sign2idx)
for name in CACHE_SUBSETS:
    subset = get_subset(name)
    build_subset_cache(train_split, "train", subset, COORDS, DATA_DIR)
    build_subset_cache(val_split, "val", subset, COORDS, DATA_DIR)

total_gb = sum(f.stat().st_size for f in FEATURES_DIR.glob("*_data.npy")) / 1e9
print(f"total feature-cache size on disk: {total_gb:.1f} GB")

## 4. Training runs

One registry run folder per subset at `data/models/<run_id>/` (run_id = epoch seconds at training start), trained by `modules.model.train_run` under regime **v2-plateau-300**. **Auto-resume**: interrupting and re-running this cell continues each unfinished subset from its last epoch. **Finished runs are never reused** — re-running trains a fresh run per subset. Per-class evaluation is *not* done here — that is §6's handoff to `modules/scripts/eval_gru.py`.

In [ ]:
# ============================================================
# Train one run per subset under the current regime (tunable: RUN_SUBSETS).
# ONE progress bar per run carries all information (batch progress, losses,
# accuracies, LR, best-so-far, plateau counter). Resume-safe: re-run after
# any interrupt continues each unfinished run in place.
# NOTE: finished runs are NOT skipped — re-running this cell trains a
# fresh epoch-seconds run folder per subset, even with identical
# conditions. Trim RUN_SUBSETS if you don't want that.
# ============================================================
RUN_SUBSETS = TRAIN_SUBSETS

run_dirs = {}
for name in RUN_SUBSETS:
    run_dirs[name] = train_run(
        arch=ARCH, subset_name=name, coords=COORDS, hyp=HYP,
        regime=TRAIN_REGIME, source=SOURCE,
        notes=f"{name} subset · regime {TRAIN_REGIME}.")

print("run folders:")
for name, rd in run_dirs.items():
    print(f"  {name}: {rd.name}")

## 5. Learning curves & comparison

Reads each run **from disk** (via the pointer files), so it works in a fresh kernel without re-running §4. Saves each run's curves into its `assets/` folder (registered in `meta.json`) and tabulates train-loop val accuracy against the historical references.

In [ ]:
# ============================================================
# Learning curves + comparison table (tunable: REPORT_SUBSETS)
# Loads each run via its pointer file — no dependency on §4 live state.
# Curves are saved into each run's assets/ and registered in its meta.json.
# ============================================================
REPORT_SUBSETS = TRAIN_SUBSETS
# historical v1-regime GRU references — those run folders predate the
# 2026-07-18 registry reset (records live in git history only)
BASELINES = {"GRU FULL_543 (v1 regime, 2026-07-13)": 0.7059,
             "GRU ME_126 (v1 regime, 2026-07-15)": 0.7373}

rows = []
for name in REPORT_SUBSETS:
    rd = pointer_run_dir(ptr_key(name))
    if rd is None:
        print(f"{name}: no run yet — train it in §4 first")
        continue
    save_learning_curves(rd, title=f"{name} · run {rd.name}")
    plt.show()
    rows.append(dict(subset_requested=name, **comparison_row(rd)))

if rows:
    display(pd.DataFrame(rows)
            .sort_values("train_val_acc", ascending=False).reset_index(drop=True))
for label, acc in BASELINES.items():
    print(f"reference — {label}: {acc:.4f}")

## 6. Canonical per-class eval handoff

Prints the exact `modules/scripts/eval_gru.py` command per run — **run those yourself**; this notebook never evaluates. The eval promotes the run's `meta.json` to `eval_status: "canonical"`; rebuild `data/models/index.csv` afterwards. Leaderboard displacement requires this exact split/metric.

In [ ]:
# ============================================================
# Canonical per-class eval handoff (tunable: DOC_SUBSETS).
# meta.json is already written/updated every epoch by the training driver —
# this cell only prints the eval + index-rebuild commands for the user.
# ============================================================
DOC_SUBSETS = TRAIN_SUBSETS

print("=== canonical per-class eval — run these from the repo root (user) ===")
for name in DOC_SUBSETS:
    rd = pointer_run_dir(ptr_key(name))
    if rd is None:
        print(f"\n# {name}: no run yet")
        continue
    finished = load_meta(rd)["training"]["finished"]
    print(f"\n# {name} · run {rd.name}"
          + ("" if finished else "  (WAIT: training incomplete)"))
    print(eval_command(rd))

print("\nafterwards, rebuild the index:")
print(".venv/Scripts/python.exe src/modules/scripts/build_model_index.py")

## 7. TFLite export chain — DEFERRED (Kaggle submission step)

Run **only after** §6's per-class eval has picked the submission model, then set `EXPORT_RUN_ID` below. ONNX → TF SavedModel → TFLite with the grader's exact `serving_default` convention (`inputs (T, 543, 3)` → `outputs (250,)`).

Notes:
1. NaN→0 and the landmark-subset gather live **inside the exported graph**, so the grader's raw full-543 frames (NaNs included) work unchanged.
2. The wrapper feeds the *full* sequence to the GRU (no ≥128-frame subsample), same as the original baseline export — but training/eval subsample longer sequences, so verify parity on long videos before submitting.
3. The ONNX export uses **`dynamo=False`** (legacy TorchScript exporter): it maps `nn.GRU` to the native ONNX `GRU` op with a genuinely dynamic frame axis, whereas torch ≥2.9's default dynamo exporter can't trace the recurrence as a dynamic loop (risking specialization to the dummy length) and additionally requires `onnxscript`.
4. Dependencies `onnx`, `onnxruntime`, `onnx2tf` are declared in `pyproject.toml` alongside `tensorflow` — `uv sync` if imports fail; never ad-hoc `uv pip install`.
5. Export output goes to the run folder's `export/` subfolder (gitignored).

In [ ]:
# ============================================================
# (Deferred) Export config + subset-aware inference wrapper + ONNX
# Tunable: EXPORT_RUN_ID — a finished run id (epoch seconds) from §4.
# ============================================================
import numpy as np
import torch
import torch.nn as nn

from modules.model import CKPT_BEST, ROWS_PER_FRAME
from modules.model.architectures import build_model

EXPORT_RUN_ID = None   # e.g. 1789012345 — set after the canonical eval

assert EXPORT_RUN_ID, ("set EXPORT_RUN_ID to a finished run first — "
                       "§7 is the deferred Kaggle-submission step")
export_run = MODELS_DIR / str(EXPORT_RUN_ID)
EXPORT_DIR = export_run / "export"
EXPORT_DIR.mkdir(exist_ok=True)
print(f"exporting run: {export_run}")

ck = torch.load(export_run / CKPT_BEST, map_location="cpu", weights_only=False)
assert ck.get("coords", "xyz") == "xyz", (
    "grader feeds (T, 543, 3) xyz — xy-only checkpoints need a custom wrapper")
base = build_model(ck.get("arch", "gru"), ck["feature_dim"],
                   len(ck["sign2idx"]), ck["hyp"])
base.load_state_dict(ck["model_state"])
base.eval()


class InferenceWrapper(nn.Module):
    """Grader calling convention: raw (T, 543, 3) frames in (NaNs included),
    (250,) scores out. NaN cleanup + landmark selection live INSIDE the graph
    so the exported model owns its preprocessing."""

    def __init__(self, base_model, landmark_rows):
        super().__init__()
        self.base = base_model
        self.register_buffer("rows", torch.as_tensor(landmark_rows, dtype=torch.long))

    def forward(self, frames):  # (T, 543, 3)
        x = torch.nan_to_num(frames, nan=0.0, posinf=0.0, neginf=0.0)
        x = x[:, self.rows, :].reshape(1, frames.shape[0], -1)  # (1, T, F)
        x = self.base.input_norm(x)
        out, _ = self.base.gru(x)  # single full-length sequence:
        return self.base.head(out[:, -1]).squeeze(0)  # no packing -> ONNX-friendly


infer_model = InferenceWrapper(base, ck["landmarks"]).eval()

# parity check: wrapper forward must equal the training-style packed forward
with torch.no_grad():
    dummy = torch.randn(48, ROWS_PER_FRAME, 3)
    ref = base(dummy[:, ck["landmarks"], :].reshape(1, 48, -1), torch.tensor([48]))
    got = infer_model(dummy)
    assert torch.allclose(ref.squeeze(0), got, atol=1e-5), "wrapper != base forward"
print("wrapper parity OK — exporting ONNX")

# dynamo=False: keep the legacy TorchScript exporter (deprecated but present).
# It maps nn.GRU to the native ONNX GRU op with a truly dynamic frame axis —
# the dynamo/torch.export path can't trace the data-dependent recurrence as a
# dynamic loop (and would pull in onnxscript). Same path as the baseline export.
torch.onnx.export(
    infer_model,
    torch.randn(30, ROWS_PER_FRAME, 3),
    str(EXPORT_DIR / "gru_model.onnx"),
    input_names=["inputs"],
    output_names=["outputs"],
    dynamic_axes={"inputs": {0: "frames"}},
    opset_version=13,
    dynamo=False,
)
print("ONNX export complete:", EXPORT_DIR / "gru_model.onnx")

In [ ]:
# ============================================================
# (Deferred) ONNX -> TF SavedModel, wrapped with the grader's
# serving_default signature. Requires onnx2tf + tensorflow (see §7 notes).
# Re-declares EXPORT_DIR from EXPORT_RUN_ID — independent of the ONNX cell.
# ============================================================
import subprocess

EXPORT_DIR = MODELS_DIR / str(EXPORT_RUN_ID) / "export"

result = subprocess.run(
    ["onnx2tf", "-i", str(EXPORT_DIR / "gru_model.onnx"),
     "-o", str(EXPORT_DIR / "saved_model_dir"), "-osd"],
    capture_output=True, text=True)
print(result.stdout[-2000:])
print(result.stderr[-2000:])

import tensorflow as tf

saved_model = tf.saved_model.load(str(EXPORT_DIR / "saved_model_dir"))


class ServingModule(tf.Module):
    def __init__(self, sm):
        super().__init__()
        self.sm = sm

    @tf.function(input_signature=[
        tf.TensorSpec(shape=[None, 543, 3], dtype=tf.float32, name="inputs")])
    def serving_default(self, inputs):
        out = self.sm.signatures["serving_default"](inputs=inputs)
        return {"outputs": next(iter(out.values()))}


serving = ServingModule(saved_model)
tf.saved_model.save(serving, str(EXPORT_DIR / "final_saved_model"),
                    signatures={"serving_default": serving.serving_default})
print("final SavedModel written:", EXPORT_DIR / "final_saved_model")

In [ ]:
# ============================================================
# (Deferred) TFLite conversion + validation with the grader's exact
# calling convention (NaNs in the input included).
# ============================================================
import numpy as np
import tensorflow as tf

from modules.model import ROWS_PER_FRAME

EXPORT_DIR = MODELS_DIR / str(EXPORT_RUN_ID) / "export"
NUM_CLASSES = 250

converter = tf.lite.TFLiteConverter.from_saved_model(str(EXPORT_DIR / "final_saved_model"))
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # helps with the 40MB cap
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,  # GRU dynamic-loop ops may need the fallback
]
tflite_model = converter.convert()
tflite_path = EXPORT_DIR / "model.tflite"
tflite_path.write_bytes(tflite_model)
print(f"model.tflite: {tflite_path.stat().st_size / 1e6:.2f} MB (competition cap: 40 MB)")

interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
sigs = list(interpreter.get_signature_list())
assert "serving_default" in sigs, f"missing serving_default in {sigs}"
runner = interpreter.get_signature_runner("serving_default")

dummy = np.random.randn(38, ROWS_PER_FRAME, 3).astype(np.float32)
dummy[np.random.rand(*dummy.shape) < 0.1] = np.nan  # grader data contains NaN
out = runner(inputs=dummy)["outputs"]
assert out.shape[-1] == NUM_CLASSES, out.shape
print("output shape:", out.shape, "· argmax:", int(np.argmax(out)))
print("TFLite validation OK — grader calling convention reproduced")